In [5]:
from pyspark.sql import SparkSession

In [12]:
spark = SparkSession.builder\
        .appName("CoinAnalysis")\
        .master("local[*]")\
        .config("spark.sql.extensions","IO.delta.sql.DeltaSparkSession")\
        .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")\
        .getOrCreate()

In [13]:
raw = spark.read.csv("Summary_by_items.csv",header=True,inferSchema=True)
print(f"raw rows:{raw.count()}")

raw rows:20123


In [14]:
from pyspark.sql import functions as F
money_columns = [ "Gross Demand Sales",
    "Gross AOV",
    "Gross Cost",
    "Cancel Amt",
    "Return Amt",
    "Exchange Amt",
    "Net Marketing Sales",
    "Net COGs",
    "Net Margin",
    "Net AOV",
    "Net Item Cost",
    "Adv Cost"]

In [23]:
cleaned = raw 
for col in money_columns: 
    cleaned = cleaned.withColumn(col,
                                 F.regexp_replace(F.col(col),r'[\$,\(\)]','').cast("float")
                      .withColumn("Margin Rate",
                (F.col("Net Margin") / F.col("Net Marketing Sales") * 100)
                .cast("decimal(10,2)")) \
                .filter(F.col("Net Units") > 0)  \
                .withColumn("Margin Rate",
                (F.col("Net Margin") / F.col("Net Marketing Sales") * 100)
                .cast("decimal(10,2)")) 
            #    .filter(F.col("Net Units") > 0)

_IncompleteInputError: incomplete input (1311900607.py, line 12)

In [ ]:
cleaned.select("description","composition","Net AOV","Gross Demand Sales").show(5)

In [18]:
cleaned = cleaned \
    .withColumn("Margin Rate",
                (F.col("Net Margin") / F.col("Net Marketing Sales") * 100)
                .cast("decimal(10,2)")) \
                .filter(F.col("Net Units") > 0) \
                .withcolumn("Net Units","Net Marketing Sales","Margin Rate")

{"ts": "2026-04-28 20:59:39.484", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.BINARY_OP_WRONG_TYPE] Cannot resolve \"(Net Margin / Net Marketing Sales)\" due to data type mismatch: the binary operator requires the input type (\"DOUBLE\" or \"DECIMAL\"), not \"STRING\". SQLSTATE: 42K09", "context": {"file": "line 3 in cell [18]", "line": "", "fragment": "__truediv__", "errorClass": "DATATYPE_MISMATCH.BINARY_OP_WRONG_TYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o76.withColumn.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_WRONG_TYPE] Cannot resolve \"(Net Margin / Net Marketing Sales)\" due to data type mismatch: the binary operator requires the input type (\"DOUBLE\" or \"DECIMAL\"), not \"STRING\". SQLSTATE: 42K09;\n'Project [Cust ##239, CODIV#240, Channel#241, Order Number#242, Order Date#243, Gross Ord#244, Net Ord#245, Gross Demand Sales#246, Gross Units#247, Gross Cost#248

AnalysisException: [DATATYPE_MISMATCH.BINARY_OP_WRONG_TYPE] Cannot resolve "(Net Margin / Net Marketing Sales)" due to data type mismatch: the binary operator requires the input type ("DOUBLE" or "DECIMAL"), not "STRING". SQLSTATE: 42K09;
'Project [Cust ##239, CODIV#240, Channel#241, Order Number#242, Order Date#243, Gross Ord#244, Net Ord#245, Gross Demand Sales#246, Gross Units#247, Gross Cost#248, Gross AOV#249, Cancel Units#250, Cancel Amt#251, Return Units#252, Return Amt#253, Exchange Units#254, Exchange Amt#255, Net Units#256, Net Marketing Sales#257, Net COGs#258, Net Margin#259, Net Margin %#260, Net AOV#261, Net Item Cost#262, Shipment Date#263, ... 21 more fields]
+- Relation [Cust ##239,CODIV#240,Channel#241,Order Number#242,Order Date#243,Gross Ord#244,Net Ord#245,Gross Demand Sales#246,Gross Units#247,Gross Cost#248,Gross AOV#249,Cancel Units#250,Cancel Amt#251,Return Units#252,Return Amt#253,Exchange Units#254,Exchange Amt#255,Net Units#256,Net Marketing Sales#257,Net CO